In [1]:
#%%writefile VideoTransformer.py
import torch
import torch.nn as nn
from torchvision import models
from torchvision import transforms
import torchvision.transforms.functional as F

from PIL import Image, ImageFilter

class VideoTransformer(nn.Module):
    def __init__(self, num_classes):
        super(VideoTransformer, self).__init__()
        # 1. バックボーン（画像から特徴抽出）
        # 例: ResNet18の最終層以外を利用 (特徴量サイズ 512)
        #resnet = models.resnet18(pretrained=True)
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        
        # 2. Transformer層
        # d_model: 特徴量次元, nhead: 注意機構のヘッド数
        encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        
        # 3. 分類用ヘッド
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        # xの形状: (Batch, Time, Channel, Height, Width)
        b, t, c, h, w = x.shape
        
        # 各フレームをバックボーンに通す
        # (B*T, C, H, W) に変換して一括処理
        x = x.view(b * t, c, h, w)
        features = self.backbone(x) # (B*T, 512, 1, 1)
        features = features.view(b, t, -1) # (Batch, Time, 512)
        
        # Transformerは (Sequence, Batch, Feature) の形状を期待する
        features = features.permute(1, 0, 2)
        
        # Transformerによる時系列処理
        out = self.transformer(features) # (Time, Batch, 512)
        
        # 最後のフレームの出力、または平均プーリングを使用して分類
        out = out.mean(dim=0) # (Batch, 512)
        return self.fc(out)



Overwriting VideoTransformer.py


In [2]:
#%%writefile AspectRatioPad.py

import torchvision.transforms.functional as F
from PIL import Image, ImageFilter

class AspectRatioPad:
    """アスペクト比を維持してリサイズし、余白を黒で埋める"""
    def __init__(self, size=(224, 224)):
        self.size = size

    def __call__(self, image):
        # 1. 元のサイズを取得
        w, h = image.size
        target_w, target_h = self.size
        
        # 2. 倍率を計算（アスペクト比維持）
        ratio = min(target_w / w, target_h / h)
        new_w, new_h = int(w * ratio), int(h * ratio)
        
        # 3. リサイズ
        image = F.resize(image, (new_h, new_w))
        
        # 4. 黒塗りの土台（キャンバス）を作成
        new_image = Image.new("RGB", self.size, (0, 0, 0))
        
        # 5. 中心に貼り付け
        upper = (target_h - new_h) // 2
        left = (target_w - new_w) // 2
        new_image.paste(image, (left, upper))
        
        return new_image

Writing AspectRatioPad.py


In [ ]:

# --- Compose でまとめる ---
custom_transform = transforms.Compose([
    AspectRatioPad(size=(224, 224)),       # アスペクト比維持＋黒埋め
    transforms.RandomHorizontalFlip(p=0.5), # 50%の確率で左右反転
    transforms.ToTensor(),
    transforms.Normalize(                  # ResNetの推奨正規化
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    ),
])

フォルダ構成の想定  
text  
  
datasets/  
  ├── dog/  
  │    ├── video01.mp4  
  │    └── video02.mp4  
  └── cat/  
       ├── video03.mp4  
       └── video04.mp4  

In [ ]:
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_video
from torchvision import models

from torch.utils.data import random_split


class VideoFolderDataset(Dataset):
    def __init__(self, root_dir, n_seconds=4, L_frames=8, classes =[],transform=None):
        self.n_seconds = n_seconds
        self.L_frames = L_frames
        self.transform = transform
        
        self.video_paths = []
        self.labels = []

        self.is_train = False
        
        # 1. サブフォルダ名からクラス名とIDの対応付けを作成
        # sortedを入れることでID（0, 1, 2...）を固定します
        #self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.classes = classes
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        # 2. 全動画ファイルのパスとラベルを取得
        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name)
            # .mp4 以外の形式も含む場合は拡張子を調整してください
            files = glob.glob(os.path.join(cls_dir, "*.mp4"))
            for f in files:
                self.video_paths.append(f)
                self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        # 動画読み込み (前回同様)
        video, _, _ = read_video(self.video_paths[idx], start_pts=0, end_pts=self.n_seconds, pts_unit='sec')
        
        # フレームのサンプリング
        total_frames = video.shape[0]
        if total_frames == 0:
            # 動画が壊れている、または短すぎる場合のダミー処理（必要に応じて）
            return torch.zeros((self.L_frames, 3, 224, 224)), self.labels[idx]

        indices = torch.linspace(0, total_frames - 1, steps=self.L_frames).long()
        sampled_video = video[indices]
        
        # 50%の確率で反転するかどうかを「この動画全体」に対して決める
        #do_flip = torch.rand(1) < 0.5
        do_flip = (torch.rand(1) < 0.5) if self.is_train else False
        
        processed_frames = []
        for frame in sampled_video:
            # 1. PIL変換 (この時点では H, W, C)
            img = Image.fromarray(frame.numpy())
            
            # 2. カスタム処理 (AspectRatioPad)
            img = AspectRatioPad(size=(224, 224))(img)
            
            # 3. 反転
            if do_flip:
                img = F.hflip(img)
                
            # 4. ここで [H, W, C] -> [C, H, W] と 0-1.0化が同時に行われる
            img = transforms.ToTensor()(img)
            
            # 5. 正規化
            img = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(img)
            
            processed_frames.append(img)
    
        return torch.stack(processed_frames), self.labels[idx]


# --- 実行コード ---

# ResNet用前処理
weights = models.ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

classes =['background','alert', 'hungry', 'log_time_no_see', 'miss']

# データセットの準備
full_dataset = VideoFolderDataset(
    root_dir="dataset_h264", # ここに親フォルダを指定
    n_seconds=4, 
    L_frames=8,
    classes=classes,
    transform=preprocess
)

# 全データセットをインスタンス化
#full_dataset = VideoFolderDataset(root_dir="datasets", n_seconds=2, L_frames=8)

print(f"検出されたクラス: {full_dataset.classes}")
print(f"データ総数: {len(full_dataset)}")

# 8割を学習、2割をテストに分割
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])


# DataLoader
# Dataloaderの作成
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)




In [ ]:
import torch.optim as optim

# 1. モデル、損失関数、最適化手法の準備
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 前のスレッドで定義したモデルをインスタンス化
# 例: 5種類の動作を分類する場合
num_classes=len(full_dataset.classes)
#model = VideoTransformer(num_classes=5).to(device)
model = VideoTransformer(num_classes=num_classes).to(device)

# 損失関数（分類問題なのでCrossEntropy）
criterion = nn.CrossEntropyLoss()

# 最適化（Transformer系はAdamやAdamWが相性良いです）
#optimizer = optim.Adam(model.parameters(), lr=1e-4)
# CNNとTransformerでパラメータを分けて設定
optimizer = optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},    # 控えめ
    {'params': model.transformer.parameters(), 'lr': 1e-4}  # 強め
])

#num_epochs = 20
num_epochs = 10

out_dir="output"
# 出力先のクラスフォルダを作成
if not os.path.exists(out_dir):
    os.makedirs(out_dir)


1. Warmup付きスケジューラの実装（PyTorch標準のみ）  


In [ ]:
from torch.optim.lr_scheduler import LambdaLR

# 設定
#num_warmup_steps = 500  # 最初の500ステップ（イテレーション）でWarmup
num_warmup_steps = len(train_loader) * 5  # 最初の500ステップ（イテレーション）でWarmup
num_training_steps = len(train_loader) * num_epochs

def lr_lambda(current_step: int):
    if current_step < num_warmup_steps:
        return float(current_step) / float(max(1, num_warmup_steps))
    # Warmup後は徐々に学習率を下げる（線形減衰）
    return max(0.0, float(num_training_steps - current_step) / float(max(1, num_training_steps - num_warmup_steps)))

scheduler = LambdaLR(optimizer, lr_lambda)


In [ ]:

# 2. 学習ループ

for epoch in range(num_epochs):
    # --- 学習フェーズ ---
    # 分割された dataset から元の VideoFolderDataset のフラグを True にする
    train_loader.dataset.dataset.is_train = True 
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # 3. スケジューラーの更新（学習率を下げる）
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()

    # --- 評価（テスト）フェーズ ---
    # 元の VideoFolderDataset のフラグを False にする
    test_loader.dataset.dataset.is_train = False
    model.eval()
    test_loss = 0.0
    correct_test = 0
    total_test = 0
    
    with torch.no_grad(): # 勾配計算をオフにしてメモリ節約
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total_test += labels.size(0)
            correct_test += predicted.eq(labels).sum().item()

    # 結果の表示
    #print(f"Epoch [{epoch+1}/{num_epochs}]")
    #print(f"  Train Loss: {train_loss/len(train_loader):.4f} | Acc: {100.*correct_train/total_train:.2f}%")
    #print(f"  Test  Loss: {test_loss/len(test_loader):.4f} | Acc: {100.*correct_test/total_test:.2f}%")
    print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Acc: {100.*correct_train/total_train:.2f}% | Test  Loss: {test_loss/len(test_loader):.4f} | Acc: {100.*correct_test/total_test:.2f}% | LR: {current_lr:.6f}")

print("学習完了！")


In [ ]:
latest_path = os.path.join(out_dir, "latest_model.pth")
# --- 5. 最後にモデルを保存 (Final Model) ---
torch.save(model.state_dict(), latest_path)
print(f"最終モデルを保存しました: {latest_path}")
